#### 02 – Preprocess GA4 Parquet Data

This notebook loads the raw daily GA4 e‑commerce Parquet extracts produced by `01_fetch_data.ipynb` and transforms them into a flat, analysis‑ready dataset. It is designed to take the nested BigQuery export schema (STRUCT and ARRAY fields) and convert it into regular columns that work well with pandas and downstream tools like Power BI.

**What this notebook does**

- Processes all raw GA4 Parquet files from `./data/raw/` in chronological order
- Flattens nested GA4 schema (STRUCTs and ARRAYs) into analysis-ready columns
- Flattens event-level fields (scalars, structs, key–value arrays) into wide columns
- Expands the GA4 items array into item-level rows (one per item) and joins them back to the event data
- Normalizes data types (nullable Int64, clean 0/1 flags) and saves processed files to `./data/processed/` for Power BI


In [12]:
import os
from pathlib import Path

import pyarrow.parquet as pq
import pandas as pd

# Project root
# Assumes this notebook lives in ./notebooks
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

RAW_DATA_DIR = "./data/raw/"
PROCESSED_DATA_DIR = "./data/processed/"

# Creates local output folder if it doesn’t exist
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

raw_path = Path(RAW_DATA_DIR)

# List all daily Parquet extracts produced by 01_fetch_data.ipynb
# Sort daily files in ascending order (chronological)
parquet_files = sorted(Path(RAW_DATA_DIR).glob("*.parquet"))
print(f"Found {len(parquet_files)} Parquet files to process")

# Inspect the first file to understand schema / columns
first_file = parquet_files[0]
print(f"First file: {first_file}")

# Load the first Parquet file
pyarrow_table = pq.read_table(first_file)

# Print schema for a quick overview of available fields
print("\nSchema Overview:")
print(pyarrow_table.schema)

# Convert to pandas DataFrame for exploratory analysis and preprocessing
df = pyarrow_table.to_pandas()

Found 92 Parquet files to process
First file: data\raw\ga4_ecom_20201101.parquet

Schema Overview:
event_date: string
event_timestamp: int64
event_name: string
event_params: list<element: struct<key: string, value: struct<double_value: double, float_value: null, int_value: int64, string_value: string>>>
  child 0, element: struct<key: string, value: struct<double_value: double, float_value: null, int_value: int64, string_value: string>>
      child 0, key: string
      child 1, value: struct<double_value: double, float_value: null, int_value: int64, string_value: string>
          child 0, double_value: double
          child 1, float_value: null
          child 2, int_value: int64
          child 3, string_value: string
event_previous_timestamp: int64
event_value_in_usd: double
event_bundle_sequence_id: int64
event_server_timestamp_offset: int64
user_id: null
user_pseudo_id: string
privacy_info: struct<ads_storage: null, analytics_storage: null, uses_transient_token: string>
  child 0

### Review and flattening strategy for GA4 schema

From the schema above, we can see that the GA4 BigQuery export uses nested data structures that do not map directly to flat pandas columns. When we load these tables into pandas, we will need to **flatten** the nested fields:

- **STRUCT** – becomes a dictionary object in each row (for example `device`, `ecommerce`).
- **ARRAY of STRUCT** – becomes a list of dictionaries in each row (for example `event_params`, `items`).

For this notebook we will organize the columns into three groups:

- **Scalar fields** at the event level, such as `event_date`, `event_name`, `user_pseudo_id`.
- **STRUCT fields** like `device`, `ecommerce`, `traffic_source`, which we will flatten one level deep into regular columns.
- **ARRAY fields**:
  - Dictionary arrays such as `event_params` and `items`, which we will convert into columns by combining the array name and key, for example `event_params_page_location` and `items_item_id`.


### Review and flattening strategy for GA4 schema

From the schema above, we can see that the GA4 BigQuery export uses nested data structures that do not map directly to flat pandas columns. When we load these tables into pandas, we will need to **flatten** the nested fields:

- **STRUCT** – becomes a dictionary object in each row (for example `device`, `ecommerce`)
- **ARRAY of STRUCT** – becomes a list of dictionaries in each row (for example `event_params`, `items`)

For this notebook we will organize the columns into three groups:

- **Scalar fields** at the event level, such as `event_date`, `event_name`, `user_pseudo_id`
- **STRUCT fields** like `device`, `ecommerce`, `traffic_source` which we will flatten one level deep into columns.
- **ARRAY fields**:
  - Key–value arrays such as `event_params` and `items`, which we will concatenate the structure and item names as columns like `event_params_page_location` and `items_item_id`


In [3]:
# Basic exploration: size, columns, sample rows, dtypes
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# List all column names from BigQuery
print("\nColumn names:")
print(df.columns.tolist())

# Preview using the first few records
print("\nFirst 3 rows:")
print(df.head(3))

# Check inferred data types (object, int64, float64, etc)
print("\nData types:")
print(df.dtypes)

Rows: 31,272
Columns: 23

Column names:
['event_date', 'event_timestamp', 'event_name', 'event_params', 'event_previous_timestamp', 'event_value_in_usd', 'event_bundle_sequence_id', 'event_server_timestamp_offset', 'user_id', 'user_pseudo_id', 'privacy_info', 'user_properties', 'user_first_touch_timestamp', 'user_ltv', 'device', 'geo', 'app_info', 'traffic_source', 'stream_id', 'platform', 'event_dimensions', 'ecommerce', 'items']

First 3 rows:
  event_date   event_timestamp       event_name                                       event_params  event_previous_timestamp  event_value_in_usd  event_bundle_sequence_id  event_server_timestamp_offset user_id      user_pseudo_id                                       privacy_info user_properties  user_first_touch_timestamp                             user_ltv                                             device                                                geo app_info                                     traffic_source   stream_id platform event

In [4]:
# GA4_SCHEMA is a manual map of which fields to keep from the nested GA4 export.
# It groups columns into:
#   - scalars: already flat event-level fields
#   - structs: nested STRUCT columns to flatten one level deep
#   - arrays: ARRAY-of-STRUCT columns (key–value and item arrays)
GA4_SCHEMA = {
    # Simple scalar columns (event-level, already flat)
    "scalars": [
        "event_date",
        "event_timestamp",
        "event_name",
        "event_previous_timestamp",
        "event_value_in_usd",
        "user_id",
        "user_pseudo_id",
        "event_bundle_sequence_id",
        "event_server_timestamp_offset",
        "stream_id",
        "platform",
    ],
    # STRUCT columns (flatten one level into columns like device_category, geo_country, ...)
    "structs": {
        "device": [
            "category",
            "is_limited_ad_tracking",
            "language",
            "mobile_brand_name",
            "mobile_marketing_name",
            "mobile_model_name",
            "mobile_os_hardware_model",
            "operating_system",
            "operating_system_version",
            "time_zone_offset_seconds",
            "vendor_id",
            "web_info_browser",
            "web_info_browser_version",
        ],
        "geo": [
            "continent",
            "sub_continent",
            "country",
            "region",
            "metro",
            "city",
        ],
        "app_info": [
            "id",
            "firebase_app_id",
            "install_source",
            "version",
        ],
        "privacy_info": [
            "ads_storage",
            "analytics_storage",
            "uses_transient_token",
        ],
        "user_ltv": [
            "revenue",
            "currency",
        ],
        "ecommerce": [
            "total_item_quantity",
            "purchase_revenue",
            "purchase_revenue_in_usd",
            "refund_value",
            "refund_value_in_usd",
            "shipping_value",
            "shipping_value_in_usd",
            "tax_value",
            "tax_value_in_usd",
            "transaction_id",
            "unique_items",
            "checkout_step",
        ],
        "traffic_source": [
            "name",
            "medium",
            "source",
            "campaign",
            "campaign_id",
            "ad_content",
            "ad_group",
        ],
        "publisher": [
            "ad_revenue_in_usd",
            "ad_format",
            "ad_source_name",
            "ad_unit_id",
        ],
    },
    # ARRAY-of-STRUCT columns (flatten key–value arrays and direct-field arrays)
    "arrays": {
        # key–value arrays: we'll transform selected keys and value to key_value column name
        # list of named attributes → becomes one column per key
        "event_params": [
            "page_location",
            "page_title",
            "page_referrer",
            "engagement_time_msec",
            "session_engaged",
            "ga_session_id",
            "ga_session_number",
            "source",
            "medium",
            "campaign",
            # any other custom event_params keys
        ],
        "user_properties": [
            "user_id",
            "first_open_time",
            "user_ltv",
            "user_creation_time",
            "user_type",
            # any other custom user_properties keys
        ],
        # direct-field arrays: each element is an item struct (flattened to items_*)
        # list of products → becomes one row per product
        "items": [
            "item_id",
            "item_name",
            "item_brand",
            "item_category",
            "item_category2",
            "item_category3",
            "item_category4",
            "item_category5",
            "item_variant",
            "price",
            "quantity",
            "coupon",
            "affiliation",
            "location_id",
            "item_list_id",
            "item_list_name",
            "promotion_id",
            "promotion_name",
            "creative_name",
            "creative_slot",
        ],
    },
}

In [7]:
# Helper functions to flatten nested GA4 fields using defined GA4_SCHEMA


def flatten_struct(df, struct_col):
    """
    Flatten one STRUCT column (e.g. device, ecommerce) into a DataFrame
    with columns defined in GA4_SCHEMA["structs"][struct_col].
    """
    # If this STRUCT column is missing, return empty frame with matching index
    if struct_col not in df.columns:
        return pd.DataFrame(index=df.index)

    # Construct column names with struct prefix e.g. device_category, device_language
    field_list = GA4_SCHEMA["structs"].get(struct_col, [])  # default to empty list if not found
    selected_columns = [f"{struct_col}_{field}" for field in field_list]

    # Keep only non-null STRUCT rows to prevent normalization error
    non_null_data = df[struct_col].dropna()
    # If all rows are null, return empty columns with correct names and index
    if non_null_data.empty:
        return pd.DataFrame(columns=selected_columns, index=df.index)

    # Flatten dicts in this STRUCT column into columns (joins nested keys with "_")
    normalized_df = pd.json_normalize(non_null_data.to_list(), sep="_").add_prefix(f"{struct_col}_")

    # Map back original row index for alignment; json_normalize started a fresh index from 0
    normalized_df.index = non_null_data.index

    # Add back rows dropped by dropna() and enforce the expected columns,
    # so shape and index match the original df
    filtered_df = normalized_df.reindex(columns=selected_columns, index=df.index)

    return filtered_df


def flatten_array_key_value(df, array_col):
    """
    Flatten GA4 key–value ARRAY columns (event_params, user_properties).
    Each row contains a list like:
    [{'key': 'page_location', 'value': {'string_value': '...'}}, ...]
    The keys listed in GA4_SCHEMA["arrays"][array_col] become columns such as
    event_params_page_location, user_properties_user_type, etc.
    """
    # If this ARRAY column is missing, return empty frame with matching index
    if array_col not in df.columns:
        return pd.DataFrame(index=df.index)

    # Construct column names by concatenating array name and key
    selected_structs = GA4_SCHEMA["arrays"].get(array_col, [])
    expected_cols = [f"{array_col}_{key}" for key in selected_structs]

    non_null_data = df[array_col].dropna()
    if non_null_data.empty:
        return pd.DataFrame(columns=expected_cols, index=df.index)

    # Each list contains multiple key–value dicts;
    # exploding creates one row per dict in the list
    exploded = non_null_data.explode().dropna()
    if exploded.empty:
        return pd.DataFrame(columns=expected_cols, index=df.index)

    normalized = pd.json_normalize(exploded.tolist(), sep="_")
    # Map back original row index for alignment; json_normalize started a fresh index from 0
    normalized.index = exploded.index

    # Retrieve the first non-null value across the value_* fields.
    # GA4 stores different data types in different value_* columns.
    normalized["extracted_value"] = (
        normalized.get("value_string_value")
        .fillna(normalized.get("value_int_value"))
        .fillna(normalized.get("value_double_value"))
        .fillna(normalized.get("value_float_value"))
    )

    # Build 2-column table (key, value) before pivoting
    temp_df = pd.DataFrame(
        {"key": normalized["key"], "value": normalized["extracted_value"]},
        index=normalized.index,
    )

    # Pivot each key → its own column, e.g. event_params_page_location
    pivoted = temp_df.pivot_table(index=temp_df.index, columns="key", values="value", aggfunc="first")
    pivoted.columns = [f"{array_col}_{col}" for col in pivoted.columns]

    # Keep only the keys defined in GA4_SCHEMA and align to the main df index
    filtered_df = pivoted.reindex(columns=expected_cols, index=df.index)

    return filtered_df


def flatten_array_direct_fields(df, array_col):
    """
    Flatten direct-field ARRAY-of-STRUCT columns (e.g. items)
    Each list of item dicts becomes one row per item, with columns defined in
    GA4_SCHEMA["arrays"][array_col] (e.g. items_item_id, items_price).
    """
    # If this ARRAY column is missing, return empty frame with matching index
    if array_col not in df.columns:
        return pd.DataFrame(index=df.index)

    #  Construct column names by concatenating array name and field e.g. items_item_id, items_price etc
    selected_structs = GA4_SCHEMA["arrays"].get(array_col, [])
    expected_cols = [f"{array_col}_{key}" for key in selected_structs]

    non_null_data = df[array_col].dropna()
    if non_null_data.empty:
        return pd.DataFrame(columns=expected_cols, index=df.index)

    # Takes the list and creates one row per item in the list whilst still sharing same list item index
    # e.g. 0 {'item_id': 'SKU_1', 'item_name': 'T-shirt', 'price': 19.99}
    #      0 {'item_id': 'SKU_2', 'item_name': 'Jeans',   'price': 49.99}
    exploded = non_null_data.explode().dropna()
    if exploded.empty:
        return pd.DataFrame(columns=expected_cols, index=df.index)

    # Turns Series of exploded dicts → Python list of dicts → into 2d table
    normalized = pd.json_normalize(exploded.tolist(), sep="_")
    # Map back original row index for alignment; json_normalize started a fresh index from 0
    normalized.index = exploded.index

    # Prefix with array name, e.g. items_item_id
    normalized.columns = [f"{array_col}_{col}" for col in normalized.columns]

    # Keep only the fields defined in GA4_SCHEMA (drop extras, add missing NaN columns)
    filtered_df = normalized.reindex(columns=expected_cols)

    return filtered_df

In [13]:
total_rows = 0  # Track row count instead len(df) to reduce memory overhead

for parquet_file in parquet_files:
    print(f"Processing: {parquet_file.name}")

    # Read the Parquet file
    df = pd.read_parquet(parquet_file)
    print(f"→ Loaded {len(df):,} rows")  # thousands separator(comma)

    # event_dfs holds multiple event-level DataFrames (scalars, structs, key–value arrays)
    # that share the same index and will be concatenated side-by-side
    event_dfs = []

    # Append scalar event-level columns (already flat)
    event_dfs.append(df[GA4_SCHEMA["scalars"]].copy())

    # Append flattened STRUCT fields (e.g. device), as new columns in same index row
    for struct_col in GA4_SCHEMA["structs"].keys():
        event_dfs.append(flatten_struct(df, struct_col))

    # Append flattened key–value ARRAYs (event_params), as new columns in same index row
    for array_col in GA4_SCHEMA["arrays"].keys():
        if array_col in ["event_params", "user_properties"]:
            event_dfs.append(flatten_array_key_value(df, array_col))

    # Concatenate all event-level pieces horizontally (adds columns, same event index)
    df_base = pd.concat(event_dfs, axis=1)

    # Flatten items struct into an item-level table (one row per item, event index may repeat)
    # Events with multiple items produce multiple rows,
    # potentially leading the dataframe to have more rows than original index
    df_items = flatten_array_direct_fields(df, "items")

    # Left join items onto events by index:
    # - duplicates event row indexes when there are multiple items
    # - keeps events with no items (item columns become NaN)
    df_concat = df_base.merge(df_items, how="left", left_index=True, right_index=True)

    # Convert BIGINT columns to nullable Int64
    numeric_cols = [
        "event_timestamp",
        "event_previous_timestamp",
        "event_bundle_sequence_id",
        "event_server_timestamp_offset",
        "device_time_zone_offset_seconds",
        "event_params_ga_session_id",
    ]
    for col in numeric_cols:
        if col in df_concat.columns:
            df_concat[col] = pd.to_numeric(df_concat[col], errors="coerce").astype("Int64")

    # Convert mixed types to string while preserving NULL
    # GA4’s session_engaged is effectively a 0/1 flag
    # Hence, normalize everything to clean string flags "0" or "1" and keeps missing values as None
    if "event_params_session_engaged" in df_concat.columns:
        df_concat["event_params_session_engaged"] = df_concat["event_params_session_engaged"].apply(
            lambda x: str(int(float(x))) if pd.notna(x) else None
        )

    # Save processed file
    output_filename = parquet_file.stem + "_processed.parquet"
    output_path = Path(PROCESSED_DATA_DIR) / output_filename
    df_concat.to_parquet(output_path, index=False)
    print(f"→ Saved {len(df_concat):,} rows to {output_filename}")

    total_rows += len(df_concat)

print(f"\n{'-'*60}")
print(f"Processing complete!")
print(f"Total files processed: {len(parquet_files)}")
print(f"Total rows processed: {total_rows:,}")

Processing: ga4_ecom_20201101.parquet
→ Loaded 31,272 rows
→ Saved 47,414 rows to ga4_ecom_20201101_processed.parquet
Processing: ga4_ecom_20201102.parquet
→ Loaded 48,388 rows
→ Saved 76,207 rows to ga4_ecom_20201102_processed.parquet
Processing: ga4_ecom_20201103.parquet
→ Loaded 61,672 rows
→ Saved 97,267 rows to ga4_ecom_20201103_processed.parquet
Processing: ga4_ecom_20201104.parquet
→ Loaded 51,866 rows
→ Saved 81,242 rows to ga4_ecom_20201104_processed.parquet
Processing: ga4_ecom_20201105.parquet
→ Loaded 51,952 rows
→ Saved 80,310 rows to ga4_ecom_20201105_processed.parquet
Processing: ga4_ecom_20201106.parquet
→ Loaded 49,004 rows
→ Saved 80,860 rows to ga4_ecom_20201106_processed.parquet
Processing: ga4_ecom_20201107.parquet
→ Loaded 34,309 rows
→ Saved 54,473 rows to ga4_ecom_20201107_processed.parquet
Processing: ga4_ecom_20201108.parquet
→ Loaded 30,137 rows
→ Saved 44,424 rows to ga4_ecom_20201108_processed.parquet
Processing: ga4_ecom_20201109.parquet
→ Loaded 44,756 ro